# 09 · Serengeti buffalo model: covariates, camera traps and map

08 found that the crater highlands and the Serengeti don't share a buffalo signal, and that a **Serengeti-only, regularized** model is the first to agree with the camera traps in a region it never saw (camera rho 0.30). This notebook builds on that model:

1. **Direct habitat covariates:** distance to rivers, dry- and wet-season greenness, tree cover, elevation and slope. Do they transfer to unseen regions better than the AlphaEarth embeddings, or add to them?
2. **A model trained on the camera traps:** cameras record effort and real non-detections, which GBIF doesn't. Does a camera-trained model rank the GBIF sightings above background? That's the reverse of the camera check.
3. **The Serengeti map** from both models, side by side.

Everything is restricted to the Serengeti ecosystem (`bv.SERENGETI_AREAS`). Data from `buffalo_data.py`, checks from `buffalo_validation.py`.

## 0. Data

The Serengeti-only dataset from 08: presences and year-matched background inside the Serengeti ecosystem's protected areas. Each point and each camera then gets six **direct covariates** (`bd.add_covariates`), with the same recipe as the embeddings: the mean inside the 700 m buffer, for the point's own year where the layer changes yearly.

| covariate | source | what it captures |
|---|---|---|
| `dist_river_m` | HydroSHEDS rivers draining ≥ 100 km² | buffalo need to drink daily |
| `ndvi_dry` | MODIS NDVI, Jul–Oct | green grazing left in the dry season |
| `ndvi_wet` | MODIS NDVI, Mar–May | productivity in the long rains |
| `tree_cover_pct` | MODIS tree cover | woodland vs open plains |
| `elevation_m`, `slope_deg` | SRTM | terrain |

Everything is cached (`cache/`), so only the first run calls Earth Engine for the covariates.

In [1]:
import ee
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

import buffalo_data as bd
import buffalo_validation as bv
from buffalo_data import BAND_COLS, COVARIATE_COLS, BuffaloConfig, build_buffalo_dataset

EE_PROJECT = "project-0cf410a3-f35f-4911-9c5"
ee.Initialize(project=EE_PROJECT)

config = BuffaloConfig(cache_dir="cache")
data = build_buffalo_dataset(config)
areas = bv.fetch_protected_areas()

points = bv.add_utm(data.sdm_df, config.cell_size_m)
points["area"] = bv.protected_area_of(points, areas)
serengeti, ratio = bv.year_matched(points[points["area"].isin(bv.SERENGETI_AREAS)])
serengeti = bd.cached(config, "serengeti_covariates", lambda: bd.add_covariates(serengeti, "point_id", config))

cam = bv.add_utm(data.cameras, config.cell_size_m)
cam = bd.cached(config, "camera_covariates", lambda: bd.add_covariates(cam, "SiteID", config))
cam = cam.merge(bd.camera_detection_days(config), on="SiteID", how="left").fillna({"detection_days": 0})
cam["detection_days"] = cam["detection_days"].clip(upper=cam["trap_days"])

print(f"Serengeti: {int(serengeti['presence'].sum())} presences, {int((serengeti['presence'] == 0).sum())} background ({ratio} per presence)")
print("missing covariates:", serengeti[COVARIATE_COLS].isna().any(axis=1).sum(), "points,", cam[COVARIATE_COLS].isna().any(axis=1).sum(), "cameras")

presences: loaded from cache\buffalo_presences_8be5ffc111.parquet
buffalo sightings in box: 674
presences after thinning: 658
mammals: loaded from cache\buffalo_mammals_8be5ffc111.parquet
mammal sightings in box: 15873
visited cells: 334


points_embedded: loaded from cache\buffalo_points_embedded_8be5ffc111.parquet
points missing embeddings: 0
cameras_embedded: loaded from cache\buffalo_cameras_embedded_8be5ffc111.parquet
dataset: 658 presences, 6580 background, 222 cameras


serengeti_covariates: loaded from cache\buffalo_serengeti_covariates_8be5ffc111.parquet
camera_covariates: loaded from cache\buffalo_camera_covariates_8be5ffc111.parquet


Serengeti: 241 presences, 2410 background (10 per presence)
missing covariates: 0 points, 0 cameras


### What the covariates look like

Median of each covariate for buffalo sightings vs background. A large difference hints at a covariate that matters. It doesn't prove it, because tourist routes also follow rivers and roads.

In [2]:
medians = serengeti.groupby("presence")[COVARIATE_COLS].median().T.rename(columns={0: "background", 1: "presences"})
medians["presences / background"] = (medians["presences"] / medians["background"]).round(2)
medians.round(2)

presence,background,presences,presences / background
dist_river_m,3257.19,3020.23,0.93
ndvi_dry,0.37,0.36,0.96
ndvi_wet,0.64,0.62,0.97
tree_cover_pct,8.05,7.51,0.93
elevation_m,1574.22,1541.64,0.98
slope_deg,2.72,2.76,1.02


## 1. Embeddings, covariates, or both?

The same check as 08, on the Serengeti data with the regularized XGBoost setting, for three feature sets:
- **embeddings:** the 64 AlphaEarth bands, as in 06–08. This row should match 08's "Serengeti only, regularized".
- **covariates:** the 6 direct layers only.
- **embeddings + covariates:** all 70.

The feature set is chosen by **region AUC** (unseen regions). Camera rho stays an independent check that is never used for choosing.

In [3]:
FEATURE_SETS = {"embeddings": BAND_COLS, "covariates": COVARIATE_COLS, "embeddings + covariates": BAND_COLS + COVARIATE_COLS}
SETTING = bv.XGB_SETTINGS["regularized"]
K_REGIONS = 4

results, region_tables = {}, {}
for name, features in FEATURE_SETS.items():
    results[name], region_tables[name] = bv.region_evaluation(serengeti, cam, K_REGIONS, SETTING, features)

comparison = pd.DataFrame(results).T.drop(columns=["camera p"]).astype(float).round(3)
best_features = comparison["region AUC"].idxmax()
print("chosen by region AUC:", best_features)
comparison

chosen by region AUC: embeddings + covariates


,block AUC,region AUC,region TSS,calibration slope,camera rho,median rate low,median rate medium,median rate high
embeddings,0.715,0.589,0.113,0.086,0.300,2.773,1.819,10.428
covariates,0.754,0.591,0.124,0.307,0.433,1.057,7.125,9.934
embeddings + covariates,0.755,0.623,0.192,0.211,0.491,0.706,3.677,13.822


### Which covariates does the model use?

Gain importance from a covariates-only model fit on all Serengeti points: how much each layer improves the trees' splits, as shares adding to 100 %. It shows what the model leans on, not cause and effect.

In [4]:
covariate_model = bv.make_model(serengeti["presence"], SETTING).fit(serengeti[COVARIATE_COLS], serengeti["presence"])
importance = pd.Series(covariate_model.get_booster().get_score(importance_type="gain")).reindex(COVARIATE_COLS).fillna(0)
(importance / importance.sum() * 100).sort_values(ascending=False).round(1).astype(str) + " %"

elevation_m       25.8 %
slope_deg         19.8 %
ndvi_dry          17.0 %
ndvi_wet          14.4 %
dist_river_m      11.7 %
tree_cover_pct    11.3 %
dtype: str

## 2. A model trained on the camera traps

Each camera ran for a known number of days, and on each day it either photographed a buffalo or it didn't. That's presence **and absence** with known effort, which GBIF never has. The model learns "chance that a camera here photographs a buffalo on a given day" directly:
- one "yes" row per camera, weighted by its detection days, and one "no" row weighted by its other trap-days
- the strongly regularized setting, because there are only 222 cameras
- no class balancing, so the output is a real daily probability

**Two checks:**
1. **Camera regions held out:** the cameras are split into 4 k-means regions, and each is predicted by a model trained on the other three. Spearman rho between predicted and observed detection rate shows whether the camera model works away from its training cameras.
2. **Reverse check on GBIF:** trained on all cameras, does it rank the GBIF buffalo sightings above background? This is reported near the cameras (within 10 km), where the camera model has data, and for the whole Serengeti, which is far outside the camera area.

In [5]:
CAMERA_FEATURES = FEATURE_SETS[best_features]
CAMERA_SETTING = bv.XGB_SETTINGS["strongly regularized"]
K_CAMERA_REGIONS = 4
NEAR_CAMERAS_M = 10_000


def fit_camera_model(c):
    X = pd.concat([c[CAMERA_FEATURES], c[CAMERA_FEATURES]], ignore_index=True)
    y = np.r_[np.ones(len(c)), np.zeros(len(c))]
    weight = np.r_[c["detection_days"], c["trap_days"] - c["detection_days"]]
    return XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=-1, **CAMERA_SETTING).fit(X, y, sample_weight=weight)


cam["camera region"] = KMeans(n_clusters=K_CAMERA_REGIONS, n_init=10, random_state=42).fit_predict(cam[["x", "y"]])
cam["held_out_p"] = np.nan
for region in range(K_CAMERA_REGIONS):
    test = cam["camera region"] == region
    cam.loc[test, "held_out_p"] = fit_camera_model(cam[~test]).predict_proba(cam.loc[test, CAMERA_FEATURES])[:, 1]

rho, p_value = spearmanr(cam["held_out_p"], cam["rate"])
print(f"camera model, camera regions held out: Spearman rho {rho:.2f} (p = {p_value:.4f}, {len(cam)} cameras)")
for region, group in cam.groupby("camera region"):
    print(f"  region {region}: {len(group)} cameras, rho {spearmanr(group['held_out_p'], group['rate'])[0]:.2f}")

camera_model = fit_camera_model(cam)
serengeti["camera_p"] = camera_model.predict_proba(serengeti[CAMERA_FEATURES])[:, 1]
distance_to_camera = cKDTree(cam[["x", "y"]]).query(serengeti[["x", "y"]])[0]
near = distance_to_camera <= NEAR_CAMERAS_M

print()
print(f"camera model on GBIF, within {NEAR_CAMERAS_M / 1000:.0f} km of a camera: AUC {roc_auc_score(serengeti.loc[near, 'presence'], serengeti.loc[near, 'camera_p']):.3f} "
      f"({int(serengeti.loc[near, 'presence'].sum())} presences, {int((serengeti.loc[near, 'presence'] == 0).sum())} background)")
print(f"camera model on GBIF, whole Serengeti: AUC {roc_auc_score(serengeti['presence'], serengeti['camera_p']):.3f}")

camera model, camera regions held out: Spearman rho 0.16 (p = 0.0142, 222 cameras)
  region 0: 57 cameras, rho 0.42
  region 1: 56 cameras, rho 0.46
  region 2: 59 cameras, rho 0.14
  region 3: 50 cameras, rho 0.25

camera model on GBIF, within 10 km of a camera: AUC 0.537 (140 presences, 377 background)
camera model on GBIF, whole Serengeti: AUC 0.419


**Reading it:**
- **Positive rho with camera regions held out:** the camera model predicts detection rates at cameras it never saw.
- **GBIF AUC above 0.5 near the cameras:** a model trained only on effort-corrected camera data, with no tourist sightings at all, still ranks tourist sightings above background. Two independent data sources agree on what buffalo habitat looks like, which is stronger evidence than either alone.
- **The whole-Serengeti AUC** is an extrapolation far beyond the ~1,100 km² camera area. A drop there is expected and says more about the camera model's coverage than about either model.

## 3. Serengeti map: GBIF model vs camera model

Both final models score the same grid: points every `GRID_SPACING_M` (2.5 km) inside the visited 10 km cells **and** inside the Serengeti ecosystem's protected areas, for `MAP_YEAR`. The grid gets embeddings and covariates with the same recipes (cached after the first run).

- **GBIF model:** XGBoost, regularized, chosen feature set, fit on all Serengeti presences and background.
- **Camera model:** fit on all cameras (section 2).

The camera model is drawn **only within `NEAR_CAMERAS_M` (10 km) of a camera**: section 2 showed it doesn't extrapolate (AUC below 0.5 on GBIF across the whole Serengeti). Their scores mean different things: relative suitability vs daily detection probability. So the map shows each as a **percentile** (0 = lowest-scoring grid point, 100 = highest). The dropdown switches between them, and only the top `SHOW_TOP_PERCENT` of points is drawn, for a cleaner map. The rank correlation between the two surfaces says how much they agree overall.

In [6]:
MAP_YEAR = 2025
GRID_SPACING_M = 2_500
GEE_CHUNK = 1_000


grid = bd.cached(config, f"serengeti_grid_{MAP_YEAR}_{GRID_SPACING_M}",
                 lambda: bv.serengeti_map_grid(data.visited_cells, areas, config, MAP_YEAR, GRID_SPACING_M, GEE_CHUNK))
grid = grid.dropna(subset=BAND_COLS + COVARIATE_COLS).reset_index(drop=True)

MAP_FEATURES = FEATURE_SETS[best_features]
gbif_model = bv.make_model(serengeti["presence"], SETTING).fit(serengeti[MAP_FEATURES], serengeti["presence"])
grid["gbif_score"] = gbif_model.predict_proba(grid[MAP_FEATURES])[:, 1]
grid["camera_p"] = camera_model.predict_proba(grid[CAMERA_FEATURES])[:, 1]
grid["gbif_percentile"] = grid["gbif_score"].rank(pct=True) * 100

grid["near_cameras"] = cKDTree(cam[["x", "y"]]).query(grid[["x", "y"]])[0] <= NEAR_CAMERAS_M
near_grid = grid[grid["near_cameras"]].copy()
near_grid["camera_percentile"] = near_grid["camera_p"].rank(pct=True) * 100

print(f"grid points scored: {len(grid)}, of which {len(near_grid)} within {NEAR_CAMERAS_M / 1000:.0f} km of a camera")
print(f"rank agreement GBIF vs camera model, whole Serengeti: Spearman rho {spearmanr(grid['gbif_score'], grid['camera_p'])[0]:.2f}")
print(f"rank agreement GBIF vs camera model, camera area:     Spearman rho {spearmanr(near_grid['gbif_score'], near_grid['camera_p'])[0]:.2f}")

serengeti_grid_2025_2500: loaded from cache\buffalo_serengeti_grid_2025_2500_8be5ffc111.parquet

grid points scored: 2693, of which 399 within 10 km of a camera
rank agreement GBIF vs camera model, whole Serengeti: Spearman rho -0.07
rank agreement GBIF vs camera model, camera area:     Spearman rho 0.20


In [7]:
import plotly.graph_objects as go

SHOW_TOP_PERCENT = 50

BLUES = [[0, "#86b6ef"], [0.5, "#2a78d6"], [1, "#0d366b"]]
LAYERS = {"GBIF model": "gbif_percentile", "camera model": "camera_percentile"}

fig = go.Figure()
for label, column in LAYERS.items():
    layer = near_grid if label == "camera model" else grid
    shown = layer[layer[column] >= 100 - SHOW_TOP_PERCENT]
    fig.add_trace(go.Scattermap(
        lat=shown["decimalLatitude"], lon=shown["decimalLongitude"], mode="markers", name=label,
        marker=dict(size=8, color=shown[column], colorscale=BLUES, cmin=100 - SHOW_TOP_PERCENT, cmax=100,
                    showscale=True, colorbar=dict(title="percentile", thickness=12)),
        customdata=shown[column], hovertemplate=label + ": percentile %{customdata:.0f}<extra></extra>",
        visible=label == "GBIF model",
    ))
fig.add_trace(go.Scattermap(lat=cam["decimalLatitude"], lon=cam["decimalLongitude"], mode="markers", name="camera traps",
                            marker=dict(size=5, color="#1f1f1f"), hoverinfo="skip"))

title = "Buffalo habitat, Serengeti {year}: {label}, top {top} % of grid points"
buttons = [dict(label=label, method="update",
                args=[{"visible": [name == label for name in LAYERS] + [True]},
                      {"title": title.format(year=MAP_YEAR, label=label, top=SHOW_TOP_PERCENT)}])
           for label in LAYERS]
fig.update_layout(
    map=dict(style="carto-positron", zoom=7.5,
             center={"lat": grid["decimalLatitude"].mean(), "lon": grid["decimalLongitude"].mean()}),
    updatemenus=[dict(buttons=buttons, x=0, y=1.08, xanchor="left")],
    title=title.format(year=MAP_YEAR, label="GBIF model", top=SHOW_TOP_PERCENT),
    legend=dict(x=0.01, y=0.01), height=700, margin=dict(t=80, l=0, r=0, b=0),
)
fig.show()

**Reading the map:**
- **Where both models agree**, the habitat signal is supported by two independent data sources: tourist sightings and effort-corrected camera detections.
- **GBIF-only hotspots** may still carry tourist-route bias. The camera layer covers only the area around the cameras (black dots), where the camera model has data.
- Percentiles rank grid points against each other. "Top 50 %" doesn't mean "half the Serengeti is good buffalo habitat".